In [1]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.4.4     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
msp_spark_3_path <-  "/u/project/pasaniuc/aflynnca/projects/20240422_SPARK_rfmix_test/rfmix_out/chr3.msp.tsv"
col_names <- readLines(msp_spark_3_path, n = 2)[2]
col_names <- strsplit(col_names, "\t")[[1]]
col_names[1] <- substring(col_names[1], 2) 

In [4]:
msp_spark_3 <- read.table(msp_spark_3_path, header = FALSE, sep = "\t", col.names = col_names)

In [6]:
snp_chunk <- msp_spark_3[msp_spark_3$spos <= 62495388 & msp_spark_3$epos >= 62495388, ]


In [10]:
snp_chunk[,1:10]

,chm,spos,epos,sgpos,egpos,n.snps,SF0000003_SP0000003.0,SF0000003_SP0000003.1,SF0030253_SP0000006.0,SF0030253_SP0000006.1
,<int>,<int>,<int>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>
689,3,62419217,62598385,81.67,82.04,100,0,0,0,0


In [11]:
# Get all column names
col_names <- colnames(snp_chunk)



In [12]:
# Find columns that end in .0 or .1
col_0 <- grep("\\.0$", col_names, value = TRUE)
col_1 <- grep("\\.1$", col_names, value = TRUE)

# Strip suffix to get base names
base_names_0 <- sub("\\.0$", "", col_0)
base_names_1 <- sub("\\.1$", "", col_1)

# Find common base names (i.e., ones that have both .0 and .1)
shared_base <- intersect(base_names_0, base_names_1)



In [13]:
# Initialize named vector to store sums
sum_vector <- setNames(numeric(length(shared_base)), shared_base)

# Calculate sums for each base
for (base in shared_base) {
  col0 <- paste0(base, ".0")
  col1 <- paste0(base, ".1")
  sum_vector[base] <- sum(snp_chunk[[col0]] + snp_chunk[[col1]], na.rm = TRUE)
}

# Convert to data frame with ID and SUM as columns
sum_df <- data.frame(
  ID = names(sum_vector),
  SUM = as.numeric(sum_vector),
  stringsAsFactors = FALSE
)

head(sum_df)

,ID,SUM
,<chr>,<dbl>
1,SF0000003_SP0000003,0
2,SF0030253_SP0000006,0
3,SF0031322_SP0000007,0
4,SF0013567_SP0000009,0
5,SF0024927_SP0000014,0
6,SF0020550_SP0000015,0


In [14]:
table(sum_df$SUM)


    0     1     2 
61797  3924  2595 

In [15]:
write.csv(sum_df, "spark_rs1452075_pos_counts.csv")